# Create Schmidt Sciences Awards (FELLOWSHIP PATTERN, method-5 big-page-inline)

Schmidt Sciences (formerly Schmidt Futures, rebranded 2024) awardees from the foundation's own /awardees/ listing at schmidtsciences.org. The entire 451-row corpus renders inline in one ~580KB HTML response — method-5 static HTML, big-page-inline variant, same as Stockholm Water Prize (priority 100).

**Prerequisites:**
- Run `scripts/local/schmidt_sciences_to_s3.py` first.

**Data source:** https://www.schmidtsciences.org/awardees/ (single page, no pagination, no JS, no auth)
**S3 location:** `s3a://openalex-ingest/awards/schmidt_sciences/schmidt_sciences_awardees.parquet`

**Awarding body:**
- funder_id: 4026159580
- display_name: "Schmidt Futures" (canonical OpenAlex name; legal entity rebranded to "Schmidt Sciences" in 2024 — same funder_id, same Crossref ID. CONACYT/CONAHCYT precedent at priority 83 for name-change-without-renumber.)
- country: US
- ROR: none
- DOI: 10.13039/100027426

**Coverage (from 2026-05-22 local `--skip-upload` run):**
- 451 awardee rows across 16 known years (2016-2025) + 77 rows with no year published
- 100% grantee_name / slug coverage; 99.3% program; 99.1% focus_area; 82.9% start_year
- 451/451 unique funder_award_id (`schmidt-{year}-{program-slug}-{name-slug}`)
- 5 focus areas: Science Systems (272), AI & Advanced Computing (123), Biosciences (19), Climate (19), Astrophysics & Space (14)
- 21 distinct programs; top 5: AI in Sci (179), AI2050 (62), Science of Trustworthy AI (60), Israeli Women's Postdoctoral Award (60), Schmidt Science Polymaths (34)

**Amount/currency:** NULL with §6.7 waiver. Schmidt publishes program-level amount narrative on each program's overview page (e.g., AI2050 awards range $150k-$2M, Schmidt Science Polymaths get $2.5M total over five years) but does not publish per-grantee amounts on the awardee listing. Fellowship precedent: HHMI #44, CIFAR #79, Damon Runyon #73, Packard #95, Rita Allen #107.

**Provenance:** `schmidt_sciences_awardees` (direct from the foundation's own site, not an aggregator).

**Priority:** 108 (UCOP at 106 and Rita Allen at 107 are the immediately-prior slots in flight; Vilcek at 105 is the current main registry head).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.schmidt_sciences_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/schmidt_sciences/schmidt_sciences_awardees.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.schmidt_sciences_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.schmidt_sciences_raw;


## Step 1.5: Inspect Raw Data, Money Scan, Native Key

Schmidt is non-monetary in our schema (§6.7 waiver, per HHMI/CIFAR/Damon Runyon/Packard/Rita Allen precedent — the foundation publishes program-level amount narrative but not per-grantee amounts on the listing). The money-field discovery scan is run for completeness; we expect zero matches.

In [ ]:
%sql
-- Runbook §1.5 money-field discovery scan (expected: no matches; waivered)
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'schmidt_sciences_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'schmidt_sciences_raw'
  AND LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT grantee_name, slug, program, focus_area_slug, focus_area_display,
       start_year, end_year, year_raw, grantee_url
FROM openalex.awards.schmidt_sciences_raw LIMIT 5;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.schmidt_sciences_raw;


In [ ]:
%sql
SELECT
    COALESCE(focus_area_display, focus_area_slug, '(NULL)') AS focus_area,
    COUNT(*) AS rows
FROM openalex.awards.schmidt_sciences_raw
GROUP BY 1 ORDER BY rows DESC;


In [ ]:
%sql
SELECT
    COALESCE(program, '(NULL)') AS program,
    COUNT(*) AS rows
FROM openalex.awards.schmidt_sciences_raw
GROUP BY 1 ORDER BY rows DESC LIMIT 15;


In [ ]:
%sql
SELECT
    CASE WHEN start_year IS NULL THEN '(no year)' ELSE start_year END AS yr,
    COUNT(*) AS rows
FROM openalex.awards.schmidt_sciences_raw
GROUP BY yr ORDER BY yr;


In [ ]:
%sql
-- Multi-year vs single-year breakdown
SELECT
    CASE
        WHEN year_raw IS NULL OR year_raw = '' THEN '(no year)'
        WHEN year_raw LIKE '%,%' THEN 'multi-year range'
        ELSE 'single-year'
    END AS year_shape,
    COUNT(*) AS rows
FROM openalex.awards.schmidt_sciences_raw
GROUP BY year_shape ORDER BY rows DESC;


## Step 1.6: Informational — Schmidt Funder Lookup in `openalex.common.funder`

Schmidt Futures (`F4026159580`) is a non-F4320* funder. The Databricks `openalex.common.funder` dim only mirrors F4320* Crossref-registered funders, so this lookup returns **0 rows by design** — that is **not** an error and Step 2 does **not** depend on it.

The Step 2 transform inlines the canonical Schmidt funder row as a `SELECT`-of-constants (Abel `F8651541334` + MinCiencias `F3277441329` precedent, both FIXED + CONFIRMED 2026-05-26). This cell remains only as a sanity check for the day the dim is extended to cover non-F4320* funders.

In [ ]:
%sql
-- Informational only — 0 rows expected for non-F4320* funders like Schmidt.
-- Does NOT halt the notebook; Step 2 uses inline constants, not this dim.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4026159580;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.schmidt_sciences_awards
USING delta
AS
WITH
schmidt_funder AS (
    -- INLINED canonical Schmidt funder row.
    -- Schmidt Futures (F4026159580) is a non-F4320* funder, so it is NOT present in
    -- openalex.common.funder (which only mirrors F4320* Crossref-registered funders).
    -- Selecting from that dim would silently return 0 rows and cascade through the
    -- CROSS JOIN to produce an empty awards table — the exact failure mode that hit
    -- Abel (F8651541334) and MinCiencias (F3277441329) on 2026-05-26, both FIXED
    -- + CONFIRMED with this same inline-constants pattern.
    SELECT
        4026159580 AS funder_id,
        'Schmidt Futures' AS display_name,
        CAST(NULL AS STRING) AS ror_id,
        '10.13039/100027426' AS doi,
        'US' AS country_code
),

raw_prepared AS (
    SELECT
        *,
        TRY_CAST(start_year AS INT) AS parsed_start_year,
        TRY_CAST(end_year AS INT) AS parsed_end_year,
        TRY_TO_DATE(CONCAT(start_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(CONCAT(end_year, '-12-31'), 'yyyy-MM-dd') AS parsed_end_date
    FROM openalex.awards.schmidt_sciences_raw
    WHERE grantee_name IS NOT NULL
      AND TRIM(grantee_name) != ''
      AND funder_award_id IS NOT NULL
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.grantee_name as display_name,

        -- No bio in the listing (would require detail-page fetch); NULL for MVP.
        CAST(NULL AS STRING) as description,

        f.funder_id,
        g.funder_award_id,

        CAST(NULL AS DOUBLE) as amount,    -- §6.7 waiver
        CAST(NULL AS STRING) as currency,  -- §6.7 waiver

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'fellowship' as funding_type,

        COALESCE(
            CONCAT('Schmidt Sciences ', NULLIF(TRIM(g.program), '')),
            'Schmidt Sciences'
        ) as funder_scheme,

        'schmidt_sciences_awardees' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        g.parsed_start_year as start_year,
        g.parsed_end_year as end_year,

        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                CAST(NULL AS STRING) as name,
                CAST(NULL AS STRING) as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.grantee_url as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN schmidt_funder f
)

SELECT * FROM awards_transformed;

## Step 3: Insert Into `openalex_awards_raw` With Priority 108

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'schmidt_sciences_awardees' AND priority = 108;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    108 as priority  -- Schmidt Sciences priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.schmidt_sciences_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.schmidt_sciences_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.schmidt_sciences_awards;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    COUNT(landing_page_url) as has_url,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_amount,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) as pct_start_date,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) as pct_pi,
    ROUND(try_divide(COUNT(landing_page_url), COUNT(*)) * 100.0, 1) as pct_landing
FROM openalex.awards.schmidt_sciences_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, end_year, funder_scheme,
       lead_investigator.given_name AS pi_given,
       lead_investigator.family_name AS pi_family,
       landing_page_url
FROM openalex.awards.schmidt_sciences_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.schmidt_sciences_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.schmidt_sciences_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 20;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.schmidt_sciences_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.schmidt_sciences_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 15;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.schmidt_sciences_awards;
